# LanceDB Backend — Embedded Zero-Infrastructure Vector Search

Medha 0.3.1 ships with `LanceDBBackend`: the **only Medha backend that requires
no external services**. LanceDB is an embedded columnar vector database that
writes directly to the local filesystem (or cloud object storage: S3, GCS,
Azure Blob) with no server process, no port, and no Docker.

## Why LanceDB?

- **Zero-infra** — just a directory on disk (or a cloud URI)
- **Persistent by default** — data survives process restarts
- **Columnar storage** — efficient scans and analytics
- **Cloud-native** — `s3://`, `gs://`, `az://` URIs work out of the box
- **Ideal for CI/CD, edge, and local analytics**

This notebook covers:
1. **Embedded local mode** — store + search + CacheHit
2. **Persistence** — close and reopen, entries survive
3. **Distance metrics** — `cosine`, `l2`, `dot`
4. **Cloud S3 mode** — `lancedb_uri="s3://..."` (placeholder)
5. **GCS / Azure Blob** — `gs://` and `az://` (placeholder)
6. **TTL + `expire()`** — SQL filter on `expires_at`
7. **`stats()` + `dedup_collection()`** — bulk operations
8. **Backend comparison** — LanceDB vs InMemory, pgvector

**Requirements:**
```bash
pip install "medha-archai[lancedb,fastembed]"
# No Docker. No port. Just pip.
```

## Prerequisites

```bash
pip install "medha-archai[lancedb,fastembed]"
```

No Docker, no running server, no open port — just a local directory.
LanceDB writes columnar Parquet-like files to `lancedb_uri`.

In [ ]:
import asyncio
import shutil
import time

from medha import Medha, Settings
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

try:
    from medha import LanceDBBackend
    HAS_LANCEDB = True
    print("lancedb package is installed")
except ImportError:
    HAS_LANCEDB = False
    print("lancedb not found — install with: pip install medha-archai[lancedb]")

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

LANCE_URI = "./demo_lancedb"

## Cell 1: Embedded Local Mode — Store 5 Entries + Search + Print CacheHit

In [ ]:
if not HAS_LANCEDB:
    print("Skipping — lancedb not installed.")
else:
    shutil.rmtree(LANCE_URI, ignore_errors=True)  # clean slate

    settings = Settings(
        backend_type="lancedb",
        lancedb_uri=LANCE_URI,
        lancedb_table_prefix="demo",
    )

    pairs = [
        ("How many users are there?",      "SELECT COUNT(*) FROM users"),
        ("List all active users",          "SELECT * FROM users WHERE status = 'active'"),
        ("Average salary by department",   "SELECT dept, AVG(salary) FROM employees GROUP BY dept"),
        ("Top 10 products by revenue",     "SELECT product_id, SUM(amount) FROM orders GROUP BY product_id ORDER BY SUM(amount) DESC LIMIT 10"),
        ("Orders placed today",            "SELECT COUNT(*) FROM orders WHERE DATE(created_at) = CURDATE()"),
    ]

    async with Medha("lance_demo", embedder=embedder, settings=settings) as medha:
        for q, sql in pairs:
            await medha.store(q, sql)

        print(f"Stored {len(pairs)} entries. Backend: {type(medha._backend).__name__}")
        print(f"Data directory: {LANCE_URI}/\n")

        hit = await medha.search("Total number of registered users")
        print(f"strategy   : {hit.strategy.value}")
        print(f"confidence : {hit.confidence:.4f}")
        print(f"query      : {hit.generated_query}")

## Cell 2: Persistence — Close and Reopen, Entries Survive

The filesystem is the durable state. Re-opening Medha with the same `lancedb_uri`
recovers all previously stored entries — no server restart needed.

In [ ]:
if not HAS_LANCEDB:
    print("Skipping — lancedb not installed.")
else:
    settings = Settings(
        backend_type="lancedb",
        lancedb_uri=LANCE_URI,
        lancedb_table_prefix="demo",
    )

    # --- Session 1 already ran above; reopen to prove persistence ---
    print("Reopening LanceDB (simulating process restart)...")
    async with Medha("lance_demo", embedder=embedder, settings=settings) as medha:
        count = await medha._backend.count("lance_demo")
        print(f"Entries after reopen: {count}")

        hit = await medha.search("How many active members are there?")
        print(f"strategy={hit.strategy.value}  query={hit.generated_query!r}")

## Cell 3: Distance Metrics — `cosine`, `l2`, `dot`

| Metric | `lancedb_metric` | Score formula | Best for |
|---|---|---|---|
| Cosine similarity | `"cosine"` | `1 - cosine_distance` | **Normalized text embeddings (default)** |
| Euclidean L2 | `"l2"` | `1 / (1 + l2_distance)` | Image embeddings, unnormalized vectors |
| Dot product | `"dot"` | `-dot_distance` | Custom fine-tuned models |

**Recommendation:** use `"cosine"` (default) for all text-based embedders
(FastEmbed, OpenAI, Cohere, Gemini) — they produce normalized vectors where
cosine similarity is well-calibrated.

In [ ]:
if not HAS_LANCEDB:
    print("Skipping — lancedb not installed.")
else:
    stored_q = "How many users are there?"
    search_q = "How many users are in the system?"  # high similarity with stored_q
    sql      = "SELECT COUNT(*) FROM users"

    print(f"  {'Metric':<8s}  {'Strategy':<18s}  {'Confidence':>10s}")
    print("  " + "-" * 42)

    for metric in ["cosine", "l2", "dot"]:
        uri = f"./demo_lancedb_{metric}"
        shutil.rmtree(uri, ignore_errors=True)
        settings = Settings(
            backend_type="lancedb",
            lancedb_uri=uri,
            lancedb_metric=metric,
        )
        async with Medha(f"metric_{metric}", embedder=embedder, settings=settings) as m:
            await m.store(stored_q, sql)
            hit = await m.search(search_q)
            conf = f"{hit.confidence:.4f}" if hit.confidence else "   n/a"
            print(f"  {metric:<8s}  {hit.strategy.value:<18s}  {conf:>10s}")
        shutil.rmtree(uri, ignore_errors=True)

    print("\ncosine is recommended for normalized text embeddings.")

## Cell 4: Cloud S3 Mode (Placeholder — do not execute)

LanceDB transparently reads and writes to S3, GCS, and Azure Blob by changing
`lancedb_uri`. All other Medha API calls remain identical.

In [ ]:
# ⚠️ Placeholder — requires AWS credentials configured in env or ~/.aws/credentials
settings_s3 = Settings(
    backend_type="lancedb",
    lancedb_uri="s3://my-bucket/medha-cache",
    lancedb_table_prefix="prod",
)

print(f"lancedb_uri : {settings_s3.lancedb_uri}")
print("Requires: AWS credentials (env vars or ~/.aws/credentials)")
print("Usage is identical to local mode — only the URI changes.")
print("(Not connecting — placeholder values)")

## Cell 5: GCS / Azure Blob (Placeholders)

```python
# Google Cloud Storage
Settings(backend_type="lancedb", lancedb_uri="gs://my-bucket/medha-cache")
# Requires: GOOGLE_APPLICATION_CREDENTIALS or Workload Identity

# Azure Blob Storage
Settings(backend_type="lancedb", lancedb_uri="az://my-container/medha-cache")
# Requires: AZURE_STORAGE_ACCOUNT + AZURE_STORAGE_ACCESS_KEY (or AZURE_STORAGE_SAS_TOKEN)
```

## Cell 6: TTL + `expire()` — SQL Filter on `expires_at`

In [ ]:
if not HAS_LANCEDB:
    print("Skipping — lancedb not installed.")
else:
    uri = "./demo_lancedb_ttl"
    shutil.rmtree(uri, ignore_errors=True)

    settings = Settings(
        backend_type="lancedb",
        lancedb_uri=uri,
    )

    async with Medha("lance_ttl", embedder=embedder, settings=settings) as medha:
        await medha.store(
            "Today's revenue",
            "SELECT SUM(amount) FROM orders WHERE DATE(created_at) = CURDATE()",
            ttl=3600,   # expires in 1 hour
        )
        await medha.store(
            "Expiring soon",
            "SELECT NOW()",
            ttl=1,      # expires immediately
        )

        await asyncio.sleep(2)

        # LanceDB backend filters expires_at via SQL WHERE clause
        removed = await medha.expire()
        print(f"Removed {removed} expired entries via SQL filter on expires_at")

    shutil.rmtree(uri, ignore_errors=True)

## Cell 7: `stats()` + `dedup_collection()` — Bulk Operations

In [ ]:
if not HAS_LANCEDB:
    print("Skipping — lancedb not installed.")
else:
    uri = "./demo_lancedb_dedup"
    shutil.rmtree(uri, ignore_errors=True)

    settings = Settings(
        backend_type="lancedb",
        lancedb_uri=uri,
        collect_stats=True,
    )

    async with Medha("lance_dedup", embedder=embedder, settings=settings) as medha:
        # Store the same entry twice — creates duplicates (same query_hash)
        for _ in range(2):
            await medha.store("User count", "SELECT COUNT(*) FROM users")
        await medha.store("Active users", "SELECT * FROM users WHERE active = true")

        count_before = await medha._backend.count("lance_dedup")
        print(f"Count before dedup : {count_before}")

        dupes = await medha.dedup_collection()
        count_after = await medha._backend.count("lance_dedup")
        print(f"Duplicates removed  : {dupes}")
        print(f"Count after dedup   : {count_after}")

        # Stats
        await medha.search("How many users?")
        await medha.search("Something unrelated")
        stats = await medha.stats()
        print(f"\nStats: requests={stats.total_requests}  hit_rate={stats.hit_rate:.2%}  backend_count={stats.backend_count}")

    shutil.rmtree(uri, ignore_errors=True)

## LanceDB vs InMemoryBackend vs pgvector

| Feature | InMemoryBackend | LanceDB | pgvector |
|---|---|---|---|
| **Persistence** | No — lost on exit | **Yes** — filesystem | Yes — PostgreSQL |
| **External service** | None | None | PostgreSQL required |
| **Cloud storage** | No | S3 / GCS / Azure Blob | No |
| **Search** | Linear O(n) | ANN (disk-based) | IVFFlat / HNSW |
| **Best collection size** | < 10k | < 10M | < 100M |
| **SQL queries on data** | No | Limited (Arrow) | Full SQL |
| **Best for** | Tests, CI | **CI/CD, edge, local analytics** | PostgreSQL teams |

**Choose LanceDB when:**
- You need **persistence without running a server**
- You're deploying to edge or serverless environments
- You want to back the cache with S3/GCS/Azure for multi-instance sharing
- You're prototyping locally and want data to survive restarts

For arbitrary SQL queries on your cache data, `pgvector` is more flexible.
For the highest throughput at scale, `qdrant` or `vectorchord` are better choices.